In [40]:
import jax
import jax.numpy as jnp
import optax

from pickle import load

In [3]:
data_path = '/Users/mariana/Documents/projects/Huawei/tdsurv/data/aids-seqs.pkl'

In [6]:
data = load(open(data_path, 'rb'))

In [8]:
data.keys()

dict_keys(['seqs', 'cs', 'ts', 'cols'])

In [11]:
seqs = data['seqs']
cs = data['cs']
ts = data['ts']

In [18]:
seqs[2]

array([[ 0.        ,  1.        ,  1.        ,  1.        , -0.61058817],
       [ 0.        ,  1.        ,  1.        ,  1.        , -0.57296897],
       [ 0.        ,  1.        ,  1.        ,  1.        ,  0.01102591],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]])

In [19]:
ts[2]

3

In [20]:
jnp.eye(3)

Array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

In [32]:
def pad_to(x1, x2):
    a1, a2 = x1.shape
    b1, b2 = x2.shape
    assert b2 >= a2 and b1 >= a1

    miss_cols = b2 - a2
    miss_rows = b1 - a1

    res = jnp.hstack((x1, jnp.zeros((a1, miss_cols))))
    res = jnp.vstack((res, jnp.zeros((miss_rows, b2))))
    return res

In [67]:
def get_target_and_mask(seq, t, c, landmark=False):
    target = jnp.zeros_like(seq)
    if not c:
        target = jnp.eye(t)[::-1]
        target = pad_to(target, seq)
    if landmark:
        mask = jnp.tril(jnp.ones_like(seq), -(t-1)).astype(jnp.bool_)[::-1]
    else:
        mask = jnp.ones((1, t))
        mask = pad_to(mask, seq).astype(jnp.bool_)
    return target, mask

In [83]:
def tree_get_target_and_mask(tree, landmark=False):
    seq, t, c = tree
    
    target = jnp.zeros_like(seq)
    if not c:
        target = jnp.eye(t)[::-1]
        target = pad_to(target, seq)
    if landmark:
        mask = jnp.tril(jnp.ones_like(seq), -(t-1)).astype(jnp.bool_)[::-1]
    else:
        mask = jnp.ones((1, t))
        mask = pad_to(mask, seq).astype(jnp.bool_)
    return target, mask

In [75]:
target, mask = get_target_and_mask(seqs[1], ts[1], cs[1], False)

In [76]:
target

Array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [77]:
mask

Array([[ True,  True,  True, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False],
       [False, False, False, False, False]], dtype=bool)

In [90]:
type(seqs)

numpy.ndarray

In [81]:
optax.sigmoid_binary_cross_entropy(jnp.array(seqs[1]), target) * mask

Array([[0.6931472, 0.6931472, 0.6931472, 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       ]],      dtype=float32)

In [79]:
type(seqs)

numpy.ndarray

In [82]:
seqs.shape

(467, 5, 5)

In [88]:
get_tgt = jax.vmap(tree_get_target_and_mask)

In [95]:
ll = [seq[0], seq[0]]

In [96]:
jnp.stack(ll).shape

(2, 5)